# MPI in a Nutshell — The Beginner's MPI Toolbox

With roughly 20 of the more than 500 MPI library calls you can already write real parallel programs.
This notebook introduces the first seven of them plus two helpers.

**After working through this notebook you will be able to**

* describe the message passing paradigm and the role of the MPI standard,
* explain how an MPI program is structured, started and shut down,
* use communicators, ranks and the size of `MPI_COMM_WORLD` to distribute work,
* exchange data with blocking point-to-point communication (`MPI_Send()`, `MPI_Recv()`),
* explain how messages are matched and use wildcards, the status object and `MPI_Get_count()`,
* recognize the typical errors: mismatched messages, truncation and deadlocks,
* measure time with `MPI_Wtime()` and abort a program with `MPI_Abort()`.

Throughout the notebook you will find prompts marked with 🤔 **Predict**.
Think about the expected result *before* running the next cell — that is where most of the learning happens.

## The Message Passing Paradigm

MPI targets **distributed-memory** architectures:
each process (or processor) can only access its own, **dedicated address space** — there is no global shared address space.
Data exchange and communication between processes therefore has to be done by **explicitly passing messages** through a communication network.

![Message passing on a distributed-memory system: five nodes, each with CPU, cache, memory and network interface, are connected by a communication network; a message travels from the memory of one node to the memory of another](img/message-passing.svg)

A message passing library should

* be flexible, efficient and portable,
* hide the communication hardware and software layers from the application developer.

The widely accepted standard for message passing in HPC and numerical simulation is the **Message Passing Interface (MPI)**:

* The program is written in a **sequential language** (C or Fortran); there are no new language constructs.
* Data is exchanged between processes by sending and receiving messages via **MPI library calls**.
* Different programs on different processes are possible (*multiple programs, multiple data* — MPMD), but mostly the same program runs everywhere (SPMD, see below).

## The MPI Standard

* The **MPI Forum** defines the MPI standard, i.e. the interfaces of the library routines.
* Latest standard: **MPI 5.0** (2025), 1137 pages.
* The forum has about 60 members:
    * application developers,
    * research institutes and computing centers,
    * manufacturers of supercomputers and software vendors.
* There are successful free implementations (MPICH, MVAPICH, Open MPI) as well as vendor libraries (Intel, Cray/HPE, …).
* All documents are available at [https://www.mpi-forum.org/](https://www.mpi-forum.org/).

## Goals and Scope of MPI

![MPI as a software layer: the application calls MPI, which uses drivers for InfiniBand, Ethernet or shared memory on top of the hardware](img/mpi-layers.svg)

* **Portability** is the main goal: architecture- and hardware-independent code.
* The standard defines **C and Fortran interfaces**; wrappers for many other popular programming languages are available.
* Features for supporting parallel **libraries**.
* Support for **heterogeneous environments**, e.g. clusters with compute nodes of different architectures.

## Working with this Notebook

The code examples are written in C and run directly from the notebook.
For this, a small IPython extension provides the cell magic `%%mpi`.
It writes the cell to a file, compiles it with `mpicc` and starts it with `mpirun`.
Load it once per kernel session:

In [ ]:
%load_ext ice.magic

By default, the content of an `%%mpi` cell is treated as the **body of `main()`**.
The extension wraps it with the boilerplate every MPI program needs, so the variables `rank` and `size` are always available:

```c
int rank, size;
MPI_Init(&argc, &argv);
MPI_Comm_rank(MPI_COMM_WORLD, &rank);
MPI_Comm_size(MPI_COMM_WORLD, &size);

/* ... your cell ... */

MPI_Finalize();
```

The most important options are

| Option | Meaning |
|---|---|
| `-n N` | start `N` MPI processes (default: 4) |
| `-r` | the cell is a complete program (no wrapping) |
| `-d` | display the generated source code |
| `-v` | print the compile and launch commands |
| `-t` | add timing code (printed by rank 0) |
| `-T S` | kill the program after `S` seconds (default: 60) — handy for deadlocks |
| `-o FILE` | where to write the source code |
| `-O VAR` | store the program output as a string in the Python variable `VAR` |

There is also `%%c` for plain serial C code.
Use `%mpi -h` to see all options:

In [ ]:
%mpi -h

Let's see what is generated and executed for a tiny snippet:

In [ ]:
%%mpi -d -v -n 2
printf("I am rank %d of %d\n", rank, size);

> **Note:** When working on a cluster, set the environment variable `ICE_MPI_LAUNCHER` (e.g. to `srun`) before starting Jupyter to change the launcher, and `ICE_MPI_LAUNCHER_FLAGS` for its default flags.
> All examples are also written to `material/code`, so you can compile and run them from a terminal as well.

## Architecture

An MPI program can be looked at from three angles:

![Architecture of an MPI program: the application calls the MPI library, which sits on top of threads, processes, CPUs and network hardware](img/architecture.svg)

* **Operating system view:** a set of ordinary, independently running processes.
* **Developer's view:** a library of routines (> 500, about 20 essential ones) for
    * coordination,
    * communication,
    * synchronization.
* **User's view:** the MPI *execution environment*, responsible for
    * resource allocation,
    * parallel program startup,
    * other (implementation-dependent) behavior.

## Parallel Execution in MPI

![Parallel execution in MPI: program startup launches ranks 0 to 3, which communicate point-to-point and collectively until program shutdown](img/spmd.svg)

* Processes run throughout the whole program execution.
* The MPI startup mechanism
    * launches the processes,
    * establishes a communication context — the *communicator*.
* **Point-to-point communication** happens between pairs of processes.
* **Collective communication** happens between all processes or a subgroup (barrier, reductions, scatter/gather, …).
* The program ends with a clean shutdown by MPI.

Usually, every process runs the **same program** (*single program, multiple data* — SPMD).
MPI is a *process-based* approach: **all variables are local!**
Every process has its own copy of every variable, and changing it on one process has no effect anywhere else.
Also, there is **no automatic workload distribution** — you have to decide yourself which process does what, usually based on its rank.

> 🤔 **Predict:** In the following cell, only rank 0 sets `x = 0`. What does every rank print?

In [ ]:
%%mpi -o code/nutshell-local-variables.c
int x = 42;

if (rank == 0)
    x = 0;      // only changes the copy of rank 0

printf("rank %d: x = %d\n", rank, x);

## The C Interface of MPI

* Required header file: `#include <mpi.h>`
* Bindings: `error = MPI_Xxxx(...);`
* MPI constants are all upper case, e.g. `MPI_COMM_WORLD`
* Arrays are indexed from 0

The official C++ bindings were deprecated in MPI 2.2 and removed in MPI 3.0 — from C++, simply use the C interface.
Wrappers exist for many other languages, e.g. `mpi4py` for Python.
In this course we use C.

## MPI Error Handling

* C routines **return an `int` error code** — which may be ignored.
* The return value `MPI_SUCCESS` indicates that all is fine.
* By default, MPI **aborts the parallel computation** for any other result, so you will rarely see an error code in practice.
  You can install your own error handlers, but this is not covered here.
* `MPI_Error_string()` turns an error code into a readable message.

You will see the default behavior in action in the section on point-to-point errors below.
Here, we just look at the messages belonging to a few error codes:

In [ ]:
%%mpi -n 1
int codes[] = { MPI_SUCCESS, MPI_ERR_RANK, MPI_ERR_TRUNCATE, MPI_ERR_TAG };
char msg[MPI_MAX_ERROR_STRING];
int len;

for (int i = 0; i < 4; ++i) {
    MPI_Error_string(codes[i], msg, &len);
    printf("error code %2d: %s\n", codes[i], msg);
}

## Initialization and Finalization

The details of the MPI startup are implementation defined.

The **first** MPI call in a program initializes the parallel machine:
```c
int MPI_Init(int *argc, char ***argv);
```

The **last** MPI call shuts it down cleanly:
```c
int MPI_Finalize();
```

* No MPI calls are allowed before `MPI_Init()` (with very few exceptions) or after `MPI_Finalize()`.
* Only rank 0 of `MPI_COMM_WORLD` is guaranteed to return from `MPI_Finalize()` — do all output and cleanup before calling it.
* `stdout`/`stderr` of each MPI process are *usually* redirected to the console where the program was started — but many options are possible, depending on the implementation.

## World Communicator and Rank

`MPI_Init()` defines the communicator `MPI_COMM_WORLD`, comprising all processes that were started:

![The communicator MPI_COMM_WORLD containing eight processes with ranks 0 to 7](img/communicator.svg)

* A **communicator** defines a set of processes (`MPI_COMM_WORLD`: all of them).
* The **rank** identifies each process within a communicator.
  It runs from `0` to `size - 1`.
* One process may have different ranks if it belongs to different communicators.

Obtain the rank of the calling process:
```c
int rank;
MPI_Comm_rank(MPI_COMM_WORLD, &rank);
```

Obtain the number of processes in a communicator:
```c
int size;
MPI_Comm_size(MPI_COMM_WORLD, &size);
```

Rank and size are the basis for distributing work.
Here, each process sums up its own chunk of the numbers `0 … N-1`.

> 🤔 **Predict:** Which chunks do the processes get for `-n 3`? Change the cell and check.

In [ ]:
%%mpi -n 1 -o code/nutshell-partial-sums.c
const int N = 1000;

// determine the chunk [start, end) of this process
int start =  rank      * N / size;
int end   = (rank + 1) * N / size;

long sum = 0;
for (int i = start; i < end; ++i)
    sum += i;

printf("rank %d of %d: [%4d, %4d) -> partial sum %ld\n", rank, size, start, end, sum);

Try other numbers of processes with `-n` as well — including `-n 1`.
Each process only knows its own partial sum; to combine them, the processes have to **communicate** (see the exercises at the end).

## MPI "Hello World!" in C

Here is a complete MPI program.
The `-r` option tells the magic not to add any boilerplate:

In [ ]:
%%mpi -r -o code/nutshell-hello.c
#include <stdio.h>
#include <mpi.h>

int main(int argc, char **argv) {
    int rank, size;

    MPI_Init(&argc, &argv);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);

    printf("Hello World! I am %d of %d\n", rank, size);

    MPI_Finalize();
    return 0;
}

* `MPI_Init()` gets **pointers** to the original `argc` and `argv` — never forget the `&`!
* A communicator is required for (almost) all MPI calls.

> 🤔 **Predict:** Run the cell several times. Will the lines always appear in the same order?

### Order of the Output

The processes run independently, and the launcher forwards their output as it arrives.
There is **no guarantee about the order** of lines from different ranks — not even that the output order reflects the order in which things happened.
Don't use `printf()` to reason about the timing of events across processes!

In [ ]:
%%mpi -n 3
for (int i = 0; i < 3; ++i)
    printf("rank %d: line %d\n", rank, i);

Most launchers can prefix each line with the rank it came from:

| Launcher | Option |
|---|---|
| Open MPI `mpirun` | `--output tag` (older versions: `--tag-output`) |
| MPICH / Intel MPI `mpirun` | `-prepend-rank` |
| Slurm `srun` | `--label` |

The following Python cell picks the right option for your launcher, and the `%%mpi` cell passes it on via `-L` (IPython replaces `$tag_flags` by the value of the Python variable; the `=` keeps flags starting with `-` from being taken as options of the magic).
Note that `-L` replaces the default launcher flags, so for Open MPI we have to repeat `--oversubscribe`:

In [ ]:
import os
from ice.magic import ICEMagic

launcher = os.environ.get('ICE_MPI_LAUNCHER', 'mpirun')
if os.path.basename(launcher) == 'srun':
    tag_flags = '--label'
elif ICEMagic.is_open_mpi(launcher):
    tag_flags = '--oversubscribe --output tag'
else:
    tag_flags = '-prepend-rank'     # MPICH, Intel MPI
tag_flags

In [ ]:
%%mpi -n 3 -L="$tag_flags"
for (int i = 0; i < 3; ++i)
    printf("rank %d: line %d\n", rank, i);

If the order matters, let one process do all the printing: the others send their results to rank 0, which receives them in a fixed order.
You will see how in the section on point-to-point communication.

## Compiling and Running the Code

**Compiling/linking**

* Headers and libraries must be found by the compiler.
* Most implementations provide *wrapper scripts* that behave like normal compilers/linkers, e.g.
    * `mpicc` / `mpicxx` (or `mpiCC`),
    * `mpiicx` for Intel MPI with the oneAPI compilers.

**Running**

* Details are implementation specific.
* Startup wrappers: `mpirun`, `mpiexec`, `srun` (Slurm), …

The same steps the magic performs can of course be done by hand.
Note how the wrapper just calls a normal compiler with the right include and library paths:

In [ ]:
%%bash
mpicc --showme 2>/dev/null || mpicc -show

mpicc -O3 -o "${TMPDIR:-/tmp}/hello" code/nutshell-hello.c
mpirun -n 4 "${TMPDIR:-/tmp}/hello"

On a cluster such as the ones at NHR@FAU, this typically looks like
```
$ mpiicx -o hello hello.c
$ srun -n 4 ./hello
Hello World! I am 3 of 4
Hello World! I am 1 of 4
Hello World! I am 0 of 4
Hello World! I am 2 of 4
```

Since the details are implementation specific, you should ask yourself:
* Where and how are the processes started?
* Can I set the process–core affinity (pinning)?
* Where does the output go?
* Do I need a shared file system?

## Point-to-Point Communication

To transfer a message, MPI needs to know

* Which process is **sending** the message?
* **Where** is the data on the sending process?
* **What kind** of data is being sent?
* **How much** data is there?
* Which process is **receiving** the message?
* **Where** should the data be left on the receiving process?
* **How much** data is the receiving process prepared to accept?

Sender and receiver must pass their part of this information to MPI **separately**.

![A message travels from the sender, rank i, to the receiver, rank j. It consists of the data (buffer, count, datatype) and an envelope (source, destination, tag, communicator)](img/message.svg)

An MPI message consists of two parts, which together answer the questions above:

* the **data**: buffer, count and data type — *where* the data is, *what kind* and *how much*,
* the **envelope**: source, destination, tag and communicator — *who* sends to *whom*, and in which context.

An MPI message is an **array of elements of a particular type**.
Data types are either
* basic (predefined), or
* MPI derived types (built from basic ones).

## Predefined Data Types in MPI (Selection)

| MPI data type | C data type |
|---|---|
| `MPI_CHAR` | `char` (printable character) |
| `MPI_SHORT` | `signed short int` |
| `MPI_INT` | `signed int` |
| `MPI_LONG` | `signed long int` |
| `MPI_LONG_LONG` | `signed long long int` |
| `MPI_UNSIGNED_CHAR` | `unsigned char` |
| `MPI_UNSIGNED_SHORT` | `unsigned short int` |
| `MPI_UNSIGNED` | `unsigned int` |
| `MPI_UNSIGNED_LONG` | `unsigned long int` |
| `MPI_INT32_T`, `MPI_INT64_T` | `int32_t`, `int64_t` |
| `MPI_UINT32_T`, `MPI_UINT64_T` | `uint32_t`, `uint64_t` |
| `MPI_FLOAT` | `float` |
| `MPI_DOUBLE` | `double` |
| `MPI_LONG_DOUBLE` | `long double` |
| `MPI_BYTE` | 8 binary digits (no conversion) |
| `MPI_PACKED` | data packed with `MPI_Pack()` |

* **Data type matching:** the same type is required in the send and the receive call.
* **Support for heterogeneous systems:** MPI performs automatic data type conversion between different architectures, if necessary.
* The width of `int` and `long` depends on the platform (`long` has 64 bit on Linux, but 32 bit on Windows).
  When values may get large, use the fixed-width types from `<stdint.h>` together with `MPI_INT64_T` and friends.
* The `count` arguments of the MPI calls are of type `int`, too.
  For more than 2³¹−1 elements, MPI 4.0 added the *large-count* variants with the suffix `_c` (e.g. `MPI_Send_c()`), which take an `MPI_Count`.

## MPI Blocking Point-to-Point Communication

* **Point-to-point:** one sender, one receiver, both identified by their rank.
* **Blocking:** after the MPI call returns,
    * the *source* process can safely modify the send buffer,
    * the receive buffer (on the *destination* process) contains the entire message.

From the MPI standard:
> **blocking** — A procedure is blocking if return from the procedure indicates the user is allowed to reuse resources specified in the call.

## Standard Blocking Send

```c
int MPI_Send(const void *buf, int count, MPI_Datatype datatype,
             int dest, int tag, MPI_Comm comm);
```

| Argument | Meaning |
|---|---|
| `buf` | address of send buffer |
| `count` | number of elements |
| `datatype` | MPI data type |
| `dest` | destination rank |
| `tag` | message tag (distinguishes different types of messages) |
| `comm` | communicator |

At completion
* the send buffer can be reused,
* the status of the destination is **unknown** — the message could be anywhere.

## Standard Blocking Receive

```c
int MPI_Recv(void *buf, int count, MPI_Datatype datatype,
             int source, int tag, MPI_Comm comm, MPI_Status *status);
```

| Argument | Meaning |
|---|---|
| `buf` | address of receive buffer |
| `count` | number of elements that fit into the receive buffer |
| `datatype` | MPI data type |
| `source` | sending process rank |
| `tag` | message tag |
| `comm` | communicator |
| `status` | address of status object (or `MPI_STATUS_IGNORE`) |

At completion
* the message has been received successfully.

Let's send an array of five `double`s from rank 0 to rank 1.

> 🤔 **Predict:** Rank 0 overwrites `data[0]` right after `MPI_Send()`. Which value does rank 1 receive?

In [ ]:
%%mpi -n 2 -o code/nutshell-send-recv.c
double data[5];

if (rank == 0) {
    for (int i = 0; i < 5; ++i)
        data[i] = 1.5 * i;

    MPI_Send(data, 5, MPI_DOUBLE, 1, 42, MPI_COMM_WORLD);

    // the send buffer may safely be reused as soon as MPI_Send() returns
    data[0] = -1.0;
    printf("rank 0: sent data\n");
} else if (rank == 1) {
    MPI_Status status;
    MPI_Recv(data, 5, MPI_DOUBLE, 0, 42, MPI_COMM_WORLD, &status);

    printf("rank 1: received");
    for (int i = 0; i < 5; ++i)
        printf(" %.1f", data[i]);
    printf("\n");
}

Point-to-point communication also solves the output-order problem from above:
every rank sends its line of text to rank 0, which receives the messages **from one specific rank after the other** and prints them.
The sender only sends the characters actually used; the `count` of the receive is just an upper bound for the message size (more on that below).

In [ ]:
%%mpi -n 4 -i string.h -o code/nutshell-ordered-output.c
char line[100];
snprintf(line, sizeof(line), "rank %d says hello", rank);

if (rank != 0) {
    MPI_Send(line, strlen(line) + 1, MPI_CHAR, 0, 0, MPI_COMM_WORLD);   // including the terminating '\0'
} else {
    printf("%s\n", line);
    for (int source = 1; source < size; ++source) {
        MPI_Recv(line, sizeof(line), MPI_CHAR, source, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        printf("%s\n", line);
    }
}

## Source and Tag Wildcards

`MPI_Recv()` accepts **wildcards** for the source and tag arguments:
`MPI_ANY_SOURCE`, `MPI_ANY_TAG`

The actual source and tag values are then available in the **status object**:
```c
MPI_Status s;
MPI_Recv(buf, count, datatype, MPI_ANY_SOURCE, MPI_ANY_TAG, MPI_COMM_WORLD, &s);
printf("Received from rank %d with tag %d\n", s.MPI_SOURCE, s.MPI_TAG);
```

In the following example, every rank except 0 sends a message to rank 0, using its own rank as tag.
Rank 0 accepts the messages in whatever order they arrive.

> 🤔 **Predict:** In contrast to the ordered output above, will the order of the lines be the same every time? Run the cell a few times.

In [ ]:
%%mpi -n 6 -o code/nutshell-wildcards.c
if (rank != 0) {
    int value = rank * rank;
    MPI_Send(&value, 1, MPI_INT, 0, rank, MPI_COMM_WORLD);
} else {
    for (int i = 1; i < size; ++i) {
        int value;
        MPI_Status s;
        MPI_Recv(&value, 1, MPI_INT, MPI_ANY_SOURCE, MPI_ANY_TAG, MPI_COMM_WORLD, &s);
        printf("Received %2d from rank %d with tag %d\n", value, s.MPI_SOURCE, s.MPI_TAG);
    }
}

## Received Message Length

The `count` argument of `MPI_Recv()` is only the **maximum** number of elements the receive buffer can hold.
The number of elements actually received can be determined from the status object:

```c
int MPI_Get_count(const MPI_Status *status, MPI_Datatype datatype, int *count);
```

| Argument | Meaning |
|---|---|
| `status` | address of status object |
| `datatype` | MPI data type |
| `count` | address of element count variable |

Here, rank `r` sends `r` integers to rank 0, which receives them into a buffer that is large enough for all of them:

In [ ]:
%%mpi -n 5 -o code/nutshell-get-count.c
#define MAX_COUNT 100

if (rank != 0) {
    int buf[MAX_COUNT];
    for (int i = 0; i < rank; ++i)
        buf[i] = rank;

    MPI_Send(buf, rank, MPI_INT, 0, 0, MPI_COMM_WORLD);
} else {
    for (int i = 1; i < size; ++i) {
        int buf[MAX_COUNT], count;
        MPI_Status s;

        MPI_Recv(buf, MAX_COUNT, MPI_INT, MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, &s);
        MPI_Get_count(&s, MPI_INT, &count);

        printf("Received %d elements from rank %d\n", count, s.MPI_SOURCE);
    }
}

## Requirements for Point-to-Point Communication

For a communication to succeed:

* the sender must specify a **valid destination** rank,
* the receiver must specify a **valid source** rank (or `MPI_ANY_SOURCE`),
* the **communicator** must be the same (e.g., `MPI_COMM_WORLD`),
* the **tags** must match (or `MPI_ANY_TAG` for the receiver),
* the message **data types** (signature) must match,
* the receiver's **buffer must be large enough**.

What happens if one of these is violated?

### Tags Do Not Match

> 🤔 **Predict:** Which of the two `printf()` lines will appear?

The magic kills the program after 5 seconds (`-T 5`).

In [ ]:
%%mpi -n 2 -T 5
int value = 42;

if (rank == 0) {
    MPI_Send(&value, 1, MPI_INT, 1, 1, MPI_COMM_WORLD);              // tag 1
    printf("rank 0: MPI_Send() returned\n");
} else if (rank == 1) {
    MPI_Recv(&value, 1, MPI_INT, 0, 2, MPI_COMM_WORLD, MPI_STATUS_IGNORE);  // tag 2
    printf("rank 1: received %d\n", value);
}

The receive never completes and the program hangs.
The *sender*, however, may well return from `MPI_Send()`: for small messages, most implementations buffer the data internally.
You cannot rely on that, though — see *When Does `MPI_Send()` Block?* below.

### Invalid Destination Rank

Ranks run from `0` to `size - 1`, so `size` itself is not a valid destination.
This error is detected, and by default MPI aborts the whole program:

In [ ]:
%%mpi -n 2
int value = 42;

if (rank == 0)
    MPI_Send(&value, 1, MPI_INT, size, 0, MPI_COMM_WORLD);   // there is no rank `size`

### Receive Buffer Too Small

If the message does not fit into the receive buffer, it is truncated.
This is an error, too:

In [ ]:
%%mpi -n 2
int data[10] = { 0 };

if (rank == 0) {
    MPI_Send(data, 10, MPI_INT, 1, 0, MPI_COMM_WORLD);   // sends 10 elements ...
} else if (rank == 1) {
    MPI_Recv(data, 5, MPI_INT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);  // ... but only 5 fit
    printf("rank 1: this line is never printed\n");
}

### Data Types Do Not Match

Rank 0 sends an `int`, rank 1 receives it as a `float`.

> 🤔 **Predict:** Will MPI detect this? What does rank 1 print?

In [ ]:
%%mpi -n 2
if (rank == 0) {
    int value = 42;
    MPI_Send(&value, 1, MPI_INT, 1, 0, MPI_COMM_WORLD);
} else if (rank == 1) {
    float value;
    MPI_Recv(&value, 1, MPI_FLOAT, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    printf("rank 1: received %g\n", value);
}

The program is erroneous, but MPI usually does **not** detect it — the bits of the integer `42` are simply reinterpreted as a `float`.
Both types have the same size here; that is why not even the length check fails.
Such bugs can be hard to find, so double-check the data types in every send/receive pair.

### When Does `MPI_Send()` Block?

The standard only guarantees that the send buffer can be reused when `MPI_Send()` returns.
How it gets there is up to the implementation:

* **small messages** are usually copied to an internal buffer and sent right away (*eager* protocol) — `MPI_Send()` returns immediately,
* **large messages** are only transferred once the receiver is ready (*rendezvous* protocol) — `MPI_Send()` blocks until a matching receive is posted.

In the following cell, both processes first send and then receive, with growing message sizes.

> 🤔 **Predict:** Will it run through? If not, at which message size does it get stuck?
> Note the last size that worked — you will compare it with the ping-pong benchmark later.

In [ ]:
%%mpi -n 2 -T 10 -o code/nutshell-send-deadlock.c
int other = 1 - rank;   // the partner process (run with exactly 2 processes)

for (int n = 1; n <= (1 << 24); n *= 4) {
    char *sendbuf = calloc(n, 1);
    char *recvbuf = malloc(n);

    MPI_Send(sendbuf, n, MPI_BYTE, other, 0, MPI_COMM_WORLD);
    MPI_Recv(recvbuf, n, MPI_BYTE, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);

    if (rank == 0)
        printf("%8d bytes exchanged\n", n);

    free(sendbuf);
    free(recvbuf);
}

Once the message size exceeds the *eager limit*, both processes wait in `MPI_Send()` for the other one to post a receive — a classic **deadlock**.
The limit depends on the MPI implementation, its configuration and the transport (shared memory within a node, network between nodes).
It usually includes a small message header: with Open MPI's default of 4 KiB for shared memory, 1024 bytes are still exchanged, but 4096 bytes already hang.
A program that relies on internal buffering is therefore **not correct**, even if it happens to work for your test case.

One way to fix it is to order the calls, so that every send meets a posted receive:

In [ ]:
%%mpi -n 2 -T 10 -o code/nutshell-send-ordered.c
int other = 1 - rank;   // the partner process (run with exactly 2 processes)

for (int n = 1; n <= (1 << 24); n *= 4) {
    char *sendbuf = calloc(n, 1);
    char *recvbuf = malloc(n);

    if (rank == 0) {
        MPI_Send(sendbuf, n, MPI_BYTE, other, 0, MPI_COMM_WORLD);
        MPI_Recv(recvbuf, n, MPI_BYTE, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    } else {
        MPI_Recv(recvbuf, n, MPI_BYTE, other, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        MPI_Send(sendbuf, n, MPI_BYTE, other, 0, MPI_COMM_WORLD);
    }

    if (rank == 0)
        printf("%8d bytes exchanged\n", n);

    free(sendbuf);
    free(recvbuf);
}

MPI offers more elegant solutions for such exchanges — `MPI_Sendrecv()` and nonblocking communication — which are not part of the beginner's toolbox.

## Beginner's MPI Toolbox

Basic point-to-point communication and support functions:

| Call | Purpose |
|---|---|
| `MPI_Init()` | let's get going |
| `MPI_Comm_size()` | how many are we? |
| `MPI_Comm_rank()` | who am I? |
| `MPI_Send()` | send data to someone else |
| `MPI_Recv()` | receive data from some-/anyone |
| `MPI_Get_count()` | how many items have I received? |
| `MPI_Finalize()` | finish off |

* Send/receive buffers may safely be reused after the call has completed.
* `MPI_Send()` must have a specific target and tag, `MPI_Recv()` does not.
* So far, there is **no explicit synchronization**!

## Some Useful MPI Calls

```c
double MPI_Wtime();
```
Returns the current time stamp (wall-clock time in seconds).
Only differences between time stamps are meaningful.
The clocks of different processes are in general **not synchronized**, so only compare time stamps taken on the same process
(unless the attribute `MPI_WTIME_IS_GLOBAL` says otherwise).
The resolution of the timer is returned by `MPI_Wtick()`.

```c
int MPI_Abort(MPI_Comm comm, int errorcode);
```
"Best effort" attempt to abort all tasks in the communicator and deliver the error code to the calling environment.
This is a **last resort** — if possible, shut down the program via `MPI_Finalize()`.

### Ping-Pong Benchmark

The following *ping-pong* benchmark uses both calls: it requires exactly two processes and aborts otherwise.
Rank 0 sends a message to rank 1, which immediately sends it back.
From the time per round trip we get the latency and bandwidth of the communication.
The first round trips of every message size are not timed — such a *warm-up* removes one-time effects like memory allocation and connection setup from the measurement.

The output is also stored in the Python variable `pingpong` (`-O pingpong`) for plotting.

In [ ]:
%%mpi -n 2 -o code/nutshell-ping-pong.c -O pingpong
if (size != 2) {
    if (rank == 0)
        fprintf(stderr, "This benchmark requires exactly 2 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

if (rank == 0)
    printf("%12s %12s %12s\n", "bytes", "time [us]", "BW [MB/s]");

for (int n = 1; n <= (1 << 22); n *= 4) {
    char *buf = calloc(n, 1);
    int reps = n <= (1 << 18) ? 10000 : 1000;

    // the first reps / 10 round trips are a warm-up and not timed
    double start = 0.0;
    for (int r = -reps / 10; r < reps; ++r) {
        if (r == 0)
            start = MPI_Wtime();

        if (rank == 0) {
            MPI_Send(buf, n, MPI_BYTE, 1, 0, MPI_COMM_WORLD);
            MPI_Recv(buf, n, MPI_BYTE, 1, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        } else {
            MPI_Recv(buf, n, MPI_BYTE, 0, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
            MPI_Send(buf, n, MPI_BYTE, 0, 0, MPI_COMM_WORLD);
        }
    }
    double time = (MPI_Wtime() - start) / reps / 2;     // one-way time per message

    if (rank == 0)
        printf("%12d %12.2f %12.1f\n", n, 1e6 * time, 1e-6 * n / time);

    free(buf);
}

In [ ]:
# plot the table printed by the benchmark above (requires matplotlib)
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator, FuncFormatter

if 'pingpong' not in globals():
    raise SystemExit('Run the ping-pong benchmark above first (it stores its output in `pingpong`).')

rows = [line.split() for line in pingpong.splitlines()]
data = [(int(r[0]), float(r[1]), float(r[2])) for r in rows if len(r) == 3 and r[0].isdigit()]
nbytes, time_us, bandwidth = zip(*data)

def human_bytes(n, _=None):
    for unit in ('B', 'KiB', 'MiB'):
        if n < 1024:
            return f'{n:g} {unit}'
        n /= 1024
    return f'{n:g} GiB'

surface, ink, muted, grid, series = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df', '#2a78d6'

fig, axes = plt.subplots(1, 2, figsize=(11, 4), facecolor=surface)
for ax, values, title in ((axes[0], time_us, 'Time per message [µs]'),
                          (axes[1], bandwidth, 'Bandwidth [MB/s]')):
    ax.set_facecolor(surface)
    ax.plot(nbytes, values, color=series, linewidth=2, marker='o', markersize=6)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.xaxis.set_major_locator(FixedLocator(nbytes[::-2]))
    ax.xaxis.set_major_formatter(FuncFormatter(human_bytes))
    ax.xaxis.set_minor_locator(FixedLocator([]))
    ax.set_title(title, loc='left', fontsize=12, color=ink)
    ax.set_xlabel('message size', color=muted)
    ax.grid(True, which='major', color=grid, linewidth=0.8)
    ax.tick_params(colors=muted, which='both')
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color('#a3a29c')

fig.tight_layout()
plt.show()

Questions:
* How does the time per message behave for small messages? For large ones?
* Why does the bandwidth saturate for large messages?
* Do you see a kink somewhere in the middle? Compare with the eager limit you found above.

Now run the benchmark code with three processes — `MPI_Abort()` stops all of them and the error code is returned to the environment:

In [ ]:
%%mpi -n 3
if (size != 2) {
    if (rank == 0)
        fprintf(stderr, "This benchmark requires exactly 2 processes, got %d\n", size);
    MPI_Abort(MPI_COMM_WORLD, 1);
}

printf("rank %d: never reached\n", rank);

### Ping-Pong on a Cluster

Here, both processes run on the same machine and communicate via shared memory.
On a cluster, it makes a big difference whether the two processes share a node or communicate over the network.
Copy `code/nutshell-ping-pong.c` to the cluster and submit a job script like the following (adapt the module names to your cluster):

```bash
#!/bin/bash -l
#SBATCH --job-name=pingpong
#SBATCH --nodes=2
#SBATCH --ntasks-per-node=2
#SBATCH --time=00:10:00

module load intel intelmpi

mpiicx -O3 -o pingpong nutshell-ping-pong.c

echo "=== inter-node: one process on each node ==="
srun --nodes=2 --ntasks=2 --ntasks-per-node=1 ./pingpong

echo "=== intra-node: both processes on the same node, pinned to cores ==="
srun --nodes=1 --ntasks=2 --cpu-bind=cores ./pingpong
```

Store the output of the job and paste it into the variable `pingpong` to plot it with the cell above.
* How do latency and bandwidth differ between the two cases?
* Within a node, does it matter whether the two cores are on the same socket? (Look at `--cpu-bind=map_cpu:...`)

## Exercises

### Exercise 1: Greetings — Write a Complete Program

So far, the magic has added `MPI_Init()` and friends for you.
This time, write the **complete** program yourself (note the `-r`):

* rank 0 sends a personal greeting such as `"Hello rank 3, greetings from rank 0!"` to every other rank,
* every other rank receives the greeting, determines its length with `MPI_Get_count()` and prints both.

Hint: `snprintf()` creates the string; send `strlen(...) + 1` characters to include the terminating `'\0'`.

<details>
<summary>Solution</summary>

```c
#include <stdio.h>
#include <string.h>
#include <mpi.h>

int main(int argc, char **argv) {
    int rank, size;
    char greeting[100];

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    if (rank == 0) {
        for (int dest = 1; dest < size; ++dest) {
            snprintf(greeting, sizeof(greeting), "Hello rank %d, greetings from rank 0!", dest);
            MPI_Send(greeting, strlen(greeting) + 1, MPI_CHAR, dest, 0, MPI_COMM_WORLD);
        }
    } else {
        int count;
        MPI_Status status;

        MPI_Recv(greeting, sizeof(greeting), MPI_CHAR, 0, 0, MPI_COMM_WORLD, &status);
        MPI_Get_count(&status, MPI_CHAR, &count);
        printf("rank %d received %d characters: %s\n", rank, count, greeting);
    }

    MPI_Finalize();
    return 0;
}
```
</details>

In [ ]:
%%mpi -r -n 4
#include <stdio.h>
#include <string.h>
#include <mpi.h>

int main(int argc, char **argv) {
    // TODO: initialize MPI, determine rank and size

    // TODO: rank 0 sends a greeting to every other rank,
    //       the other ranks receive and print it

    // TODO: shut down MPI
    return 0;
}

### Exercise 2: Global Sum

Extend the partial-sum example from above: every rank except 0 sends its partial sum to rank 0, which adds them up and prints the total.
Check your result against $N(N-1)/2$ and try different numbers of processes.

<details>
<summary>Solution</summary>

```c
const int N = 1000;

int start =  rank      * N / size;
int end   = (rank + 1) * N / size;

long sum = 0;
for (int i = start; i < end; ++i)
    sum += i;

if (rank != 0) {
    MPI_Send(&sum, 1, MPI_LONG, 0, 0, MPI_COMM_WORLD);
} else {
    long total = sum;
    for (int i = 1; i < size; ++i) {
        long partial;
        MPI_Recv(&partial, 1, MPI_LONG, MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        total += partial;
    }
    printf("total = %ld, expected %ld\n", total, (long)N * (N - 1) / 2);
}
```
</details>

In [ ]:
%%mpi -n 4
const int N = 1000;

int start =  rank      * N / size;
int end   = (rank + 1) * N / size;

long sum = 0;
for (int i = start; i < end; ++i)
    sum += i;

// TODO: collect the partial sums on rank 0 and print the total
printf("rank %d: partial sum %ld\n", rank, sum);

### Exercise 3: Token Ring

Pass a token around a ring of processes:
rank 0 starts with a token value of `0` and sends it to rank 1.
Every rank receives the token from its left neighbor `rank - 1`, adds its own rank, and passes it on to its right neighbor `(rank + 1) % size`.
Finally, rank 0 receives the token from rank `size - 1` and prints it.

> 🤔 **Predict:** Which value do you expect? What happens with `-n 1`?

<details>
<summary>Solution</summary>

```c
int token;
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;

if (rank == 0) {
    token = 0;
    MPI_Send(&token, 1, MPI_INT, right, 0, MPI_COMM_WORLD);
    MPI_Recv(&token, 1, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    printf("token = %d, expected %d\n", token, size * (size - 1) / 2);
} else {
    MPI_Recv(&token, 1, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    token += rank;
    MPI_Send(&token, 1, MPI_INT, right, 0, MPI_COMM_WORLD);
}
```

With a single process, rank 0 sends a message to itself before receiving it.
This only works because the small message is buffered internally (eager protocol) — strictly speaking, the program may deadlock.
</details>

In [ ]:
%%mpi -n 4
// TODO: pass a token around the ring

### Exercise 4: Timing and Speedup

The integral $\int_0^1 \frac{4}{1+x^2}\,dx$ equals $\pi$.
Approximate it with the midpoint rule using $N = 10^9$ intervals of width $h = 1/N$:
$$\pi \approx h \sum_{i=0}^{N-1} \frac{4}{1 + \left((i + 0.5)\,h\right)^2}$$

* Distribute the intervals across the ranks as in the partial-sum example.
  Use `int64_t` for $N$ and the loop counter: $N$ itself still fits into an `int`, but `(rank + 1) * N` overflows it from three processes on.
* Measure the time of the computation on every rank with `MPI_Wtime()` and print it.
* Collect the partial results on rank 0, which prints $\pi$ and its error.
* Run with 1, 2, 4 and 8 processes. How does the runtime scale? Compare with the `-t` option of the magic.

<details>
<summary>Solution</summary>

```c
const int64_t N = 1000000000;
const double h = 1.0 / N;

int64_t start =  rank      * N / size;
int64_t end   = (rank + 1) * N / size;

double t0 = MPI_Wtime();
double sum = 0.0;
for (int64_t i = start; i < end; ++i) {
    double x = (i + 0.5) * h;
    sum += 4.0 / (1.0 + x * x);
}
sum *= h;
double t1 = MPI_Wtime();

printf("rank %d: %.3f s\n", rank, t1 - t0);

if (rank != 0) {
    MPI_Send(&sum, 1, MPI_DOUBLE, 0, 0, MPI_COMM_WORLD);
} else {
    double pi = sum;
    for (int i = 1; i < size; ++i) {
        double partial;
        MPI_Recv(&partial, 1, MPI_DOUBLE, MPI_ANY_SOURCE, 0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
        pi += partial;
    }
    printf("pi = %.15f, error = %.2e\n", pi, pi - 3.14159265358979323846);
}
```
</details>

In [ ]:
%%mpi -n 4
// TODO: N = 10^9 intervals of width h = 1/N
// TODO: determine the chunk of this rank, compute and time its partial sum,
//       and collect the result on rank 0

### Exercise 5: Deadlock-Free Exchange in a Ring

Every rank sends a large array (`1 << 22` integers, i.e. 16 MiB) to its right neighbor `(rank + 1) % size` and receives one from its left neighbor.
If every rank first calls `MPI_Send()` and then `MPI_Recv()`, this deadlocks for large messages — just like the exchange of two processes above.
Order the calls in two different ways and compare how long the exchange takes:

* **(a)** generalize the fix for two processes from above: rank 0 sends first and then receives, all other ranks receive first and then send,
* **(b)** **even** ranks send first and then receive, **odd** ranks receive first and then send.

The cell already contains the timing code: after a short warm-up, it repeats the exchange `reps` times and prints the average time per exchange.
`MPI_Barrier()` makes all processes wait for each other before the clock is started and stopped — more on it in the notebook on collective communication.
Run the cell with 2, 3, 4, 6 and 8 processes (or as many as your machine has cores).

> 🤔 **Predict:** Let $T$ be the time to transfer one message, so that both variants take $2T$ with `-n 2`.
> How does the time per exchange grow with the number of processes in (a), and how in (b)? What changes for an odd number of processes?

<details>
<summary>Solution</summary>

Replace the `TODO` in the loop over `r` by

```c
int send_first = variant == 0 ? rank == 0 : rank % 2 == 0;

if (send_first) {
    MPI_Send(sendbuf, n, MPI_INT, right, 0, MPI_COMM_WORLD);
    MPI_Recv(recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
} else {
    MPI_Recv(recvbuf, n, MPI_INT, left,  0, MPI_COMM_WORLD, MPI_STATUS_IGNORE);
    MPI_Send(sendbuf, n, MPI_INT, right, 0, MPI_COMM_WORLD);
}
```

Both variants are deadlock-free for two or more processes, but they differ a lot in speed:

* **(a)** Only rank 0 can send right away. Rank 1 receives first and only then sends to rank 2, which has been waiting all the time, and so on.
  The messages travel around the ring one after the other, just like the token in Exercise 3: the exchange is completely **serialized** and takes $\text{size} \cdot T$.
* **(b)** With an even number of processes, all even ranks send to their odd neighbors at the same time, then all odd ranks to their even neighbors: $2T$, **independent of the number of processes**.
  With an odd number, rank `size - 1` and rank 0 are both even.
  Rank `size - 1` has to wait until rank 0 has completed its own send, and rank `size - 2` in turn has to wait for rank `size - 1`.
  This costs a third step — $3T$, but still independent of the number of processes.

For example, on an Apple M2 Pro with Open MPI (shared memory, $T \approx 0.6$ ms):

| processes | 2 | 3 | 4 | 5 | 6 | 8 |
|---|---|---|---|---|---|---|
| (a) [ms] | 1.2 | 1.8 | 2.4 | 3.1 | 3.8 | 6.4 |
| (b) [ms] | 1.2 | 1.8 | 1.7 | 2.1 | 2.1 | 3.1 |

With 3 processes, both variants need three steps.
With many processes, (b) gets slower as well: all transfers run at the same time and compete for the memory bandwidth of the node — between nodes, they would use separate network links.
With a single process, rank 0 sends a large message to itself before receiving it, and both variants deadlock.
`MPI_Sendrecv()`, introduced in the next notebook, leaves the ordering to MPI.
</details>


In [ ]:
%%mpi -n 4 -T 30
const int n = 1 << 22;      // 16 MiB per message, far above the eager limit
const int reps = 20;
int *sendbuf = malloc(n * sizeof(int));
int *recvbuf = malloc(n * sizeof(int));
int left  = (rank - 1 + size) % size;
int right = (rank + 1) % size;

for (int i = 0; i < n; ++i)
    sendbuf[i] = rank;

for (int variant = 0; variant < 2; ++variant) {
    double start = 0.0;
    recvbuf[0] = -1;

    // the first two exchanges are a warm-up and not timed
    for (int r = -2; r < reps; ++r) {
        if (r == 0) {
            MPI_Barrier(MPI_COMM_WORLD);    // all ranks start together
            start = MPI_Wtime();
        }

        // TODO: send sendbuf to the right and receive recvbuf from the left, without deadlock
        //       variant 0 (a): rank 0 sends first, all other ranks receive first
        //       variant 1 (b): even ranks send first, odd ranks receive first
    }
    MPI_Barrier(MPI_COMM_WORLD);            // the exchange is done once the last rank is done
    double time = (MPI_Wtime() - start) / reps;

    if (recvbuf[0] != left)
        printf("rank %d: variant %c did not receive the data of rank %d\n", rank, 'a' + variant, left);
    if (rank == 0)
        printf("variant %c: %.2f ms per exchange\n", 'a' + variant, 1e3 * time);
}

free(sendbuf);
free(recvbuf);

## Summary

* MPI is the standard for **message passing** on distributed-memory systems: every process has its own address space, and data is exchanged by explicit messages.
* The "parallel program" is started up with `MPI_Init()` and shut down with `MPI_Finalize()`.
* An MPI task ("process") is identified by its **rank**: `MPI_Comm_rank()`.
* The number of MPI tasks is given by `MPI_Comm_size()`.
* The startup process is very implementation dependent.
* Simple, blocking point-to-point communication with `MPI_Send()` and `MPI_Recv()`.
    * "Blocking" == the buffer can be reused as soon as the call returns.
    * `MPI_Send()` may or may not wait for the receiver — never rely on internal buffering.
* Messages are matched by source, tag and communicator; wildcards and the status object give flexibility on the receiving side.
* Typical errors: messages that never match (the program hangs), receive buffers that are too small (truncation) and deadlocks.
  Mismatched data types are usually **not** detected by MPI.
* `MPI_Wtime()` for timing, `MPI_Abort()` as a last resort.

## Outlook

Several things in this notebook were more cumbersome than necessary:

* exchanging data between pairs of processes without deadlocks (Exercise 5) → `MPI_Sendrecv()` and **nonblocking communication** (`MPI_Isend()`, `MPI_Irecv()`, `MPI_Wait()`),
* receiving a message of unknown size → `MPI_Probe()`,
* collecting partial results or distributing data → **collective communication** (`MPI_Reduce()`, `MPI_Bcast()`, `MPI_Gather()`, `MPI_Scatter()`, …), which turn Exercises 2 and 4 into a single call.

[Back to the overview](../overview.ipynb)